# 02 - Data Preprocessing

## SDG 13: Vehicle CO2 Emissions Prediction

This notebook implements the data preprocessing pipeline based on findings from `01_eda.ipynb`. The output is a clean, model-ready dataset saved to `data/processed/`.

### Pipeline Overview
1. Load raw data
2. Remove redundant columns (based on EDA findings)
3. Handle class imbalance (drop the single "N" fuel type record)
4. Split into training and test sets **before** any fitting
5. Encode categorical features (One-Hot Encoding)
6. Scale numerical features (StandardScaler)
7. Save processed datasets

### ⚠️ Key Principle: Avoiding Data Leakage

All transformations (encoders, scalers) are **fit on training data only**, then applied to both training and test sets. This prevents information from the test set leaking into the training process.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## 2. Load Raw Data

Load the original dataset and make a copy for preprocessing. Keeping the raw DataFrame untouched allows re-running this notebook from any cell.

In [2]:
df_raw = pd.read_csv('../data/raw/CO2 Emissions_Canada.csv')
df = df_raw.copy()
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (7385, 12)


,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


## 3. Clean Column Names

Replace original column names containing spaces and special characters (e.g., `Fuel Consumption Comb (L/100 km)`) with Python-friendly snake_case names.

In [3]:
df.columns = ['Make', 'Model', 'Vehicle_Class', 'Engine_Size', 'Cylinders',
              'Transmission', 'Fuel_Type', 'Fuel_City', 'Fuel_Hwy',
              'Fuel_Comb', 'Fuel_Comb_mpg', 'CO2_Emissions']

print("New column names:")
print(df.columns.tolist())

New column names:
['Make', 'Model', 'Vehicle_Class', 'Engine_Size', 'Cylinders', 'Transmission', 'Fuel_Type', 'Fuel_City', 'Fuel_Hwy', 'Fuel_Comb', 'Fuel_Comb_mpg', 'CO2_Emissions']


## 4. Handle Class Imbalance

Remove the single record with `Fuel_Type == 'N'`. With only one sample, the model cannot learn any meaningful pattern for this category, and its presence would create a near-zero-variance feature after one-hot encoding.

In [4]:
# Check before
print("Before removal:")
print(df['Fuel_Type'].value_counts())

# Remove rows where Fuel_Type is 'N'
df = df[df['Fuel_Type'] != 'N'].reset_index(drop=True)

# Check after
print("\nAfter removal:")
print(df['Fuel_Type'].value_counts())
print(f"\nNew shape: {df.shape}")

Before removal:
Fuel_Type
X    3637
Z    3202
E     370
D     175
N       1
Name: count, dtype: int64

After removal:
Fuel_Type
X    3637
Z    3202
E     370
D     175
Name: count, dtype: int64

New shape: (7384, 12)


## 5. Separate Features (X) and Target (y)

Separate the target variable (`CO2_Emissions`) from the features. This is a standard ML convention:
- **X**: Feature matrix (what the model uses to make predictions)
- **y**: Target vector (what we want to predict)

In [5]:
# Separate target from features
y = df['CO2_Emissions']
X = df.drop(columns=['CO2_Emissions'])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny sample values: {y.head().tolist()}")

X shape: (7384, 11)
y shape: (7384,)

y sample values: [196, 221, 136, 255, 244]


## 6. Drop Redundant Features

Based on the EDA findings, remove features that are either redundant (multicollinearity) or have excessive cardinality:

| Column | Reason |
|---|---|
| `Model` | 2000+ unique values, no useful encoding |
| `Make` | 40+ brands, long-tail distribution |
| `Fuel_City` | Correlation with `Fuel_Comb` = 0.99 |
| `Fuel_Hwy` | Correlation with `Fuel_Comb` = 0.98 |
| `Fuel_Comb_mpg` | Inverse of `Fuel_Comb` (L/100km) |

In [6]:
columns_to_drop = ['Model', 'Make', 'Fuel_City', 'Fuel_Hwy', 'Fuel_Comb_mpg']
X = X.drop(columns=columns_to_drop)

print(f"X shape after drop: {X.shape}")
print(f"\nRemaining columns:")
print(X.columns.tolist())

X shape after drop: (7384, 6)

Remaining columns:
['Vehicle_Class', 'Engine_Size', 'Cylinders', 'Transmission', 'Fuel_Type', 'Fuel_Comb']


## 7. Train/Test Split

Split the data into training (80%) and test (20%) sets. This MUST happen before any transformation that learns from data (like scalers and encoders) to prevent data leakage.

- `test_size=0.2` → 20% for testing
- `random_state=42` → reproducibility (anyone re-running this notebook gets the same split)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (5907, 6)
X_test shape:  (1477, 6)
y_train shape: (5907,)
y_test shape:  (1477,)


## 8. Build Preprocessing Pipeline

Use `ColumnTransformer` to apply different transformations to different column types:
- **Numerical columns** → `StandardScaler` (zero mean, unit variance)
- **Categorical columns** → `OneHotEncoder` (binary columns per category)

### Key Principle: Fit only on training data
The preprocessor will be **fit** on `X_train` only, then applied (via `transform`) to both `X_train` and `X_test`. This ensures the test set statistics never leak into the training process.

In [8]:
# Define which columns get which treatment
numerical_features = ['Engine_Size', 'Cylinders', 'Fuel_Comb']
categorical_features = ['Vehicle_Class', 'Transmission', 'Fuel_Type']

# Build the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

print("Preprocessor configured:")
print(f"  Numerical features:   {numerical_features}")
print(f"  Categorical features: {categorical_features}")

Preprocessor configured:
  Numerical features:   ['Engine_Size', 'Cylinders', 'Fuel_Comb']
  Categorical features: ['Vehicle_Class', 'Transmission', 'Fuel_Type']


## 9. Apply the Preprocessor

- `fit_transform(X_train)`: Learn the statistics (mean, std, categories) from training data AND transform it
- `transform(X_test)`: Apply those learned statistics to the test data — no learning happens here

This is the moment data leakage is prevented: the test set never influences the scaler's mean/std or the encoder's category list.

In [9]:
# Fit on training data, transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")
print(f"\nFirst row of processed training data:")
print(X_train_processed[0])

X_train_processed shape: (5907, 50)
X_test_processed shape:  (1477, 50)

First row of processed training data:
[0.61268953 0.20063795 0.59173571 0.         0.         0.
 0.         0.         1.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         1.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 1.         0.        ]


## 10. Save Processed Data

Convert the processed NumPy arrays back to DataFrames (with proper column names) and save to `data/processed/` for use in the modeling notebook.

In [10]:
# Get feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()
print(f"Total features after preprocessing: {len(feature_names)}")
print(f"\nSample feature names:")
for name in feature_names[:5]:
    print(f"  {name}")
print(f"  ...")
for name in feature_names[-3:]:
    print(f"  {name}")

Total features after preprocessing: 50

Sample feature names:
  num__Engine_Size
  num__Cylinders
  num__Fuel_Comb
  cat__Vehicle_Class_COMPACT
  cat__Vehicle_Class_FULL-SIZE
  ...
  cat__Fuel_Type_E
  cat__Fuel_Type_X
  cat__Fuel_Type_Z


In [11]:
import os

# Convert processed arrays back to DataFrames with proper column names
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

# Preview
print("X_train_df first 3 rows:")
X_train_df.head(3)

X_train_df first 3 rows:


,num__Engine_Size,num__Cylinders,num__Fuel_Comb,cat__Vehicle_Class_COMPACT,cat__Vehicle_Class_FULL-SIZE,cat__Vehicle_Class_MID-SIZE,cat__Vehicle_Class_MINICOMPACT,cat__Vehicle_Class_MINIVAN,cat__Vehicle_Class_PICKUP TRUCK - SMALL,cat__Vehicle_Class_PICKUP TRUCK - STANDARD,...,cat__Transmission_AV6,cat__Transmission_AV7,cat__Transmission_AV8,cat__Transmission_M5,cat__Transmission_M6,cat__Transmission_M7,cat__Fuel_Type_D,cat__Fuel_Type_E,cat__Fuel_Type_X,cat__Fuel_Type_Z
0,0.612690,0.200638,0.591736,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.567895,-0.889172,-0.897681,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.863042,-0.889172,-0.551305,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [12]:
# Make sure output folder exists
os.makedirs('../data/processed', exist_ok=True)

# Save X and y (both train and test)
X_train_df.to_csv('../data/processed/X_train.csv', index=False)
X_test_df.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("✅ Saved files to data/processed/:")
print("  - X_train.csv  ({} rows × {} cols)".format(*X_train_df.shape))
print("  - X_test.csv   ({} rows × {} cols)".format(*X_test_df.shape))
print("  - y_train.csv  ({} rows)".format(len(y_train)))
print("  - y_test.csv   ({} rows)".format(len(y_test)))

✅ Saved files to data/processed/:
  - X_train.csv  (5907 rows × 50 cols)
  - X_test.csv   (1477 rows × 50 cols)
  - y_train.csv  (5907 rows)
  - y_test.csv   (1477 rows)


## 11. Save the Preprocessor

In addition to the processed data, we save the fitted `preprocessor` object. This allows us to transform any new vehicle's specs in the future (e.g., during model deployment) using the exact same fitted statistics.

In [13]:
import joblib

# Make sure models folder exists
os.makedirs('../models', exist_ok=True)

# Save the fitted preprocessor
joblib.dump(preprocessor, '../models/preprocessor.joblib')

print("✅ Saved fitted preprocessor to models/preprocessor.joblib")
print(f"   File size: {os.path.getsize('../models/preprocessor.joblib')} bytes")

✅ Saved fitted preprocessor to models/preprocessor.joblib
   File size: 3441 bytes


## 12. Summary

### What This Notebook Accomplished

| Step | Action | Outcome |
|---|---|---|
| 1 | Loaded raw data | 7,385 records × 12 features |
| 2 | Renamed columns | Python-friendly `snake_case` names |
| 3 | Removed `Fuel_Type == 'N'` | 7,384 records (dropped 1 under-represented sample) |
| 4 | Separated X and y | X: features, y: CO2 Emissions |
| 5 | Dropped redundant columns | X: 11 → 6 columns |
| 6 | Train/test split (80/20) | X_train: 5907, X_test: 1477 |
| 7 | Built preprocessing pipeline | StandardScaler + OneHotEncoder via ColumnTransformer |
| 8 | Fit on training, transform both | 50-dimensional feature space after encoding |
| 9 | Saved processed data + preprocessor | Ready for modeling |

### Files Produced

- `data/processed/X_train.csv` — Processed training features (5907 × 50)
- `data/processed/X_test.csv` — Processed test features (1477 × 50)
- `data/processed/y_train.csv` — Training labels (5907)
- `data/processed/y_test.csv` — Test labels (1477)
- `models/preprocessor.joblib` — Fitted preprocessor (for reuse)

### Key Design Decisions

1. **Data leakage prevention**: All statistics (scaler means, encoder categories) learned only from `X_train`
2. **Robust encoding**: `handle_unknown='ignore'` handles any unseen categories at prediction time
3. **Reproducibility**: `random_state=42` in train/test split ensures consistent results
4. **Pipeline modularity**: `ColumnTransformer` allows different treatments for numerical vs categorical features, all bundled in a single reusable object

### Next Steps → `03_modeling.ipynb`

1. Load processed data from `data/processed/`
2. Train three regression models:
   - Linear Regression (baseline)
   - Random Forest Regressor
   - XGBoost Regressor
3. Compare performance (R², RMSE, MAE)
4. Analyze feature importance
5. Conduct ablation study (with vs. without `Fuel_Comb`) to validate model robustness